In [1]:
# ─────────────────────────── standard libs ────────────────────────────────
import math
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────── data stack ────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ────────────────────────────── sklearn ───────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)

# ─────────────────────────────── torch ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler, random_split

from tqdm import tqdm

print('All imports OK')

All imports OK


## 1. Dataset Overview

In [2]:
# ─── Load CSV ─────────────────────────────────────────────────────────────
CSV_PATH = '/kaggle/input/datasets/harunshimanto/epileptic-seizure-recognition/Epileptic Seizure Recognition.csv'

df_raw = pd.read_csv(CSV_PATH)
print(f'Raw CSV shape : {df_raw.shape}')   # (11500, 180)
print(df_raw.head(3))

Raw CSV shape : (11500, 180)
      Unnamed   X1   X2   X3   X4   X5   X6   X7   X8   X9  ...  X170  X171  \
0  X21.V1.791  135  190  229  223  192  125   55   -9  -33  ...   -17   -15   
1  X15.V1.924  386  382  356  331  320  315  307  272  244  ...   164   150   
2     X8.V1.1  -32  -39  -47  -37  -32  -36  -57  -73  -85  ...    57    64   

   X172  X173  X174  X175  X176  X177  X178  y  
0   -31   -77  -103  -127  -116   -83   -51  4  
1   146   152   157   156   154   143   129  1  
2    48    19   -12   -30   -35   -35   -36  5  

[3 rows x 180 columns]


In [3]:
# ─── Extract features and labels ──────────────────────────────────────────
# Columns: 0=row_name, 1..178=features, 179=label
# We use features at columns index 1:178 → 177 time-step values
# Label is at column index 179

values = df_raw.values  # numpy array, shape (11500, 180)

esrinput = values[0:, 1:178].astype(np.float32)   # (11500, 177)
esrlabel = values[0:, 179].astype(np.float32)      # (11500,)

print(f'Input shape : {esrinput.shape}')   # (11500, 177)
print(f'Label shape : {esrlabel.shape}')   # (11500,)

# ─── Binarise labels: 1 = seizure, 2/3/4/5 = non-seizure → 0 ─────────────
esrlabel_binary = (esrlabel == 1).astype(np.int64)  # 1 if seizure, else 0

print(f'\nClass distribution (original):')
for cls in sorted(np.unique(esrlabel)):
    print(f'  Class {int(cls)} : {(esrlabel == cls).sum():,} samples')

print(f'\nBinary labels:')
print(f'  Seizure     (1) : {esrlabel_binary.sum():,}')
print(f'  Non-seizure (0) : {(esrlabel_binary == 0).sum():,}')

Input shape : (11500, 177)
Label shape : (11500,)

Class distribution (original):
  Class 1 : 2,300 samples
  Class 2 : 2,300 samples
  Class 3 : 2,300 samples
  Class 4 : 2,300 samples
  Class 5 : 2,300 samples

Binary labels:
  Seizure     (1) : 2,300
  Non-seizure (0) : 9,200


## 2. Preprocessing

Since this CSV is already a pre-extracted feature set (no raw EDF files),  
preprocessing is limited to:
1. **StandardScaler** normalisation across all 177 features
2. **Reshape** to `(N, 1, 177)` — treating the 177 values as a single-channel time series

No bandpass filtering or resampling is needed (data is already processed).

In [4]:
# ─── StandardScaler normalisation ─────────────────────────────────────────
sc = StandardScaler()
esrinput_scaled = sc.fit_transform(esrinput)   # (11500, 177)

# ─── Reshape to (N, 1, 177): (batch, channels, time_steps) ────────────────
# We treat the 177-feature vector as a single-channel EEG segment.
X = torch.tensor(esrinput_scaled, dtype=torch.float32).unsqueeze(1)  # (11500, 1, 177)
y = torch.tensor(esrlabel_binary, dtype=torch.long)                   # (11500,)

print(f'X shape : {X.shape}')   # torch.Size([11500, 1, 177])
print(f'y shape : {y.shape}')   # torch.Size([11500])

X shape : torch.Size([11500, 1, 177])
y shape : torch.Size([11500])


## 3. Train / Val / Test Split

In [5]:
# ─── 60/20/20 split ───────────────────────────────────────────────────────
full_dataset = TensorDataset(X, y)
total_size   = len(full_dataset)

train_size = int(0.60 * total_size)
eval_size  = int(0.20 * total_size)
test_size  = total_size - train_size - eval_size

torch.manual_seed(42)
train_dataset, eval_dataset, test_dataset = random_split(
    full_dataset, [train_size, eval_size, test_size]
)

print(f'Train : {len(train_dataset):,}')
print(f'Val   : {len(eval_dataset):,}')
print(f'Test  : {len(test_dataset):,}')

# ─── DataLoaders with WeightedRandomSampler for training ──────────────────
BATCH_SIZE = 128

# Build per-sample weights for the training subset
train_labels = y[train_dataset.indices].numpy()
counts       = np.bincount(train_labels)
weights      = 1.0 / counts[train_labels]
sampler      = WeightedRandomSampler(
    torch.from_numpy(weights).float(),
    num_samples=len(weights), replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(eval_dataset,  batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

# Sanity check batch shape
xb, yb = next(iter(train_loader))
print(f'\nTrain batch X : {xb.shape}')   # torch.Size([128, 1, 177])
print(f'Train batch y : {yb.shape}')   # torch.Size([128])
print(f'Class balance : seizure={yb.sum().item()}  non={BATCH_SIZE - yb.sum().item()}')

Train : 6,900
Val   : 2,300
Test  : 2,300

Train batch X : torch.Size([128, 1, 177])
Train batch y : torch.Size([128])
Class balance : seizure=81  non=47


## 4. Model — L_SCLNet with EEGformer Backbone

Configured for:
- `input_channels = 1` (single channel)
- `time_steps = 177` (177-point feature vector treated as time series)
- `kernel_size = 5` — reduced from 10 because 177 is much shorter than 512;
  with kernel=5 the ODCM output is S = 177 − 3×(5−1) = **165** time steps.
- `num_submatrices = 6`, `num_heads_ttm = 6` (D=120 divisible by 6)

In [6]:
# ─────────────────────────────────────────────────────────────────────────
# EEGformer components
# ─────────────────────────────────────────────────────────────────────────

def trunc_normal(tensor, mean=0., std=1., a=-2., b=2.):
    def norm_cdf(x):
        return (1. + math.erf(x / math.sqrt(2.))) / 2.
    with torch.no_grad():
        l = norm_cdf((a - mean) / std)
        u = norm_cdf((b - mean) / std)
        tensor.uniform_(2 * l - 1, 2 * u - 1)
        tensor.erfinv_()
        tensor.mul_(std * math.sqrt(2.))
        tensor.add_(mean)
        tensor.clamp_(min=a, max=b)
        return tensor


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features    = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1  = nn.Linear(in_features, hidden_features)
        self.act  = act_layer()
        self.fc2  = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class GenericTFB(nn.Module):
    def __init__(self, emb_size, num_heads):
        super().__init__()
        self.M  = emb_size
        self.hA = num_heads
        self.Dh = emb_size // num_heads
        self.Wqkv = nn.Parameter(torch.randn(3, num_heads, self.Dh, emb_size))
        self.Wo   = nn.Parameter(torch.randn(emb_size, emb_size))
        self.ln1  = nn.LayerNorm(emb_size)
        self.ln2  = nn.LayerNorm(emb_size)
        self.mlp  = Mlp(emb_size, emb_size * 4)

    def forward(self, x, z):
        qkv = torch.einsum('xhdm,ijm->xijhd', self.Wqkv, self.ln1(z))
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q.transpose(1, 2) / math.sqrt(self.Dh)) @ k.transpose(1, 2).transpose(-2, -1)
        att = torch.softmax(att, dim=-1)
        imv = (att @ v.transpose(1, 2)).transpose(1, 2)
        z = torch.einsum('nm,ijm->ijn', self.Wo,
                         imv.reshape(z.shape[0], z.shape[1], self.M)) + z
        z = self.mlp(self.ln2(z)) + z
        return z


class TemporalTFB(nn.Module):
    def __init__(self, emb_size, num_heads, avgf):
        super().__init__()
        self.M    = emb_size
        self.hA   = num_heads
        self.Dh   = emb_size // num_heads
        self.avgf = avgf
        self.Wqkv = nn.Parameter(torch.randn(3, num_heads, self.Dh, emb_size))
        self.Wo   = nn.Parameter(torch.randn(emb_size, emb_size))
        self.ln1  = nn.LayerNorm(emb_size)
        self.ln2  = nn.LayerNorm(emb_size)
        self.mlp  = Mlp(emb_size, emb_size * 4)

    def forward(self, x, z):
        qkv = torch.einsum('xhdm,im->xihd', self.Wqkv, self.ln1(z))
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q.transpose(0, 1) / math.sqrt(self.Dh)) @ k.transpose(0, 1).transpose(-2, -1)
        att = torch.softmax(att, dim=-1)
        imv = (att @ v.transpose(0, 1)).transpose(0, 1)
        z = torch.einsum('nm,im->in', self.Wo,
                         imv.reshape(self.avgf + 1, self.M)) + z
        z = self.mlp(self.ln2(z)) + z
        return z


class ODCM(nn.Module):
    """
    Three-layer depthwise-conv feature extractor.
    Input  : (C, T)
    Output : (C, 120, S)  where S = T - 3*(kernel_size-1)
    """
    def __init__(self, input_channels, kernel_size=5):
        super().__init__()
        self.ncf = 120
        C = input_channels
        self.cv1  = nn.Conv1d(C, C,            kernel_size, groups=C, padding='valid')
        self.cv2  = nn.Conv1d(C, C,            kernel_size, groups=C, padding='valid')
        self.cv3  = nn.Conv1d(C, self.ncf * C, kernel_size, groups=C, padding='valid')
        self.relu = nn.ReLU()

    def forward(self, x):           # (C, T)
        x = self.relu(self.cv1(x))
        x = self.relu(self.cv2(x))
        x = self.relu(self.cv3(x))  # (ncf*C, S)
        S = x.shape[1]
        C = x.shape[0] // self.ncf
        return x.reshape(C, self.ncf, S)   # (C, D=120, S)


class RTM(nn.Module):
    def __init__(self, odcm_out_shape, num_blocks, num_heads):
        super().__init__()
        C, D, S = odcm_out_shape
        self.M  = D
        self.weight = nn.Parameter(torch.randn(D, D))
        self.bias   = nn.Parameter(torch.zeros(S, C + 1, D))
        self.cls    = nn.Parameter(torch.zeros(S, 1, D))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([GenericTFB(D, num_heads) for _ in range(num_blocks)])

    def forward(self, x):
        x = x.permute(2, 0, 1)                              # (S, C, D)
        z = torch.einsum('lm,ijm->ijl', self.weight, x)     # (S, C, D)
        z = torch.cat([self.cls, z], dim=1)                  # (S, C+1, D)
        z = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        return z


class STM(nn.Module):
    def __init__(self, rtm_out_shape, num_blocks, num_heads):
        super().__init__()
        S, Cp1, D = rtm_out_shape
        self.M  = D
        self.weight = nn.Parameter(torch.randn(D, D))
        self.bias   = nn.Parameter(torch.zeros(Cp1, S + 1, D))
        self.cls    = nn.Parameter(torch.zeros(Cp1, 1, D))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([GenericTFB(D, num_heads) for _ in range(num_blocks)])

    def forward(self, x):
        x = x.transpose(0, 1)                               # (C+1, S, D)
        z = torch.einsum('lm,ijm->ijl', self.weight, x)     # (C+1, S, D)
        z = torch.cat([self.cls, z], dim=1)                  # (C+1, S+1, D)
        z = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        return z


class TTM(nn.Module):
    def __init__(self, stm_out_shape, num_submatrices, num_blocks, num_heads):
        super().__init__()
        Cp1, Sp1, D = stm_out_shape
        self.avgf = num_submatrices
        self.M    = num_submatrices
        assert D % num_submatrices == 0
        assert self.M % num_heads == 0
        flat = Cp1 * Sp1
        self.weight = nn.Parameter(torch.randn(self.M, flat))
        self.bias   = nn.Parameter(torch.zeros(num_submatrices + 1, self.M))
        self.cls    = nn.Parameter(torch.zeros(1, self.M))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([
            TemporalTFB(self.M, num_heads, num_submatrices) for _ in range(num_blocks)
        ])
        self.ln = nn.LayerNorm(self.M)

    def forward(self, x):
        Cp1, Sp1, D = x.shape
        seg  = D // self.avgf
        flat = Cp1 * Sp1
        xc   = x.permute(2, 0, 1).reshape(self.avgf, seg, Cp1, Sp1).mean(dim=1)
        alt  = xc.reshape(self.avgf, flat)
        z    = torch.einsum('lm,im->il', self.weight, alt)
        z    = torch.cat([self.cls, z], dim=0)
        z    = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        z   = self.ln(z)
        out = torch.einsum('tm,mf->tf', z, self.weight)
        return out.reshape(self.avgf + 1, Cp1, Sp1)


class CNNDecoder(nn.Module):
    def __init__(self, ttm_out_shape, num_classes, CF_second=2):
        super().__init__()
        Mp1, Cp1, Sp1 = ttm_out_shape
        self.cvd1 = nn.Conv1d(Cp1,  1,         kernel_size=1)
        self.cvd2 = nn.Conv1d(Sp1,  CF_second,  kernel_size=1)
        self.cvd3 = nn.Conv1d(Mp1,  Mp1 // 2,   kernel_size=1)
        self.fc   = nn.Linear((Mp1 // 2) * CF_second, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.permute(2, 1, 0)
        x = self.relu(self.cvd1(x)).squeeze(1)
        x = self.relu(self.cvd2(x)).transpose(0, 1)
        x = self.relu(self.cvd3(x))
        x = self.fc(x.reshape(1, -1))
        return x


# ─── L_SCLNet with EEGformer backbone ────────────────────────────────────
class L_SCLNet_EEGformer(nn.Module):
    """
    L_SCLNet with EEGformer backbone for the Epileptic Seizure Recognition CSV.

    Key parameters vs TUH version:
        input_channels = 1   (single pseudo-channel)
        time_steps     = 177 (feature vector length)
        kernel_size    = 5   (reduced — 177 is much shorter than 512)

    Shape trace (C=1, kernel_size=5, time_steps=177):
        ODCM : S = 177 - 3*(5-1) = 165
        RTM  : (165, 2, 120)
        STM  : (2, 166, 120)
        TTM  : (7, 2, 166)  with M=6
    """
    def __init__(
        self,
        input_channels  = 1,
        time_steps      = 177,
        num_classes     = 2,
        kernel_size     = 5,    # ← reduced for 177-sample input
        num_blocks      = 3,
        num_heads_rtm   = 6,
        num_heads_stm   = 6,
        num_heads_ttm   = 6,
        num_submatrices = 6,
        CF_second       = 2,
    ):
        super().__init__()
        ncf = 120
        C   = input_channels
        D   = ncf
        S   = time_steps - 3 * (kernel_size - 1)

        assert D % num_heads_rtm == 0
        assert D % num_heads_stm == 0
        assert num_submatrices % num_heads_ttm == 0
        assert D % num_submatrices == 0

        self.odcm    = ODCM(C, kernel_size)
        self.rtm     = RTM((C, D, S),       num_blocks, num_heads_rtm)
        self.stm     = STM((S, C+1, D),     num_blocks, num_heads_stm)
        self.ttm     = TTM((C+1, S+1, D),   num_submatrices, num_blocks, num_heads_ttm)
        self.decoder = CNNDecoder((num_submatrices+1, C+1, S+1), num_classes, CF_second)

    def forward(self, x):
        # x : (B, C, T)
        outs = []
        for i in range(x.shape[0]):
            xi = self.odcm(x[i])
            xi = self.rtm(xi)
            xi = self.stm(xi)
            xi = self.ttm(xi)
            xi = self.decoder(xi)
            outs.append(xi)
        return torch.cat(outs, dim=0)   # (B, num_classes)

print('Model definition complete.')

Model definition complete.


## 5. Training

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = L_SCLNet_EEGformer(
    input_channels  = 1,
    time_steps      = 177,
    num_classes     = 2,
    kernel_size     = 5,
    num_blocks      = 3,
    num_heads_rtm   = 6,
    num_heads_stm   = 6,
    num_heads_ttm   = 6,
    num_submatrices = 6,
    CF_second       = 2,
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# Class-weighted loss
counts    = np.bincount(y[train_dataset.indices].numpy())
class_wts = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_wts)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

NUM_EPOCHS = 30
scheduler  = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1,
)

best_val_acc    = 0.0
best_model_path = 'best_lsclnet_epileptic_csv.pth'
history         = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
LOG_EPOCHS      = set(range(1, NUM_EPOCHS + 1, 10)) | {NUM_EPOCHS}

for epoch in range(1, NUM_EPOCHS + 1):
    # ── train ──────────────────────────────────────────────────────────────
    model.train()
    total_loss, all_preds, all_labels_list = 0.0, [], []
    for xb, yb in tqdm(train_loader, desc=f'Epoch {epoch:02d}', leave=False):
        xb, yb = xb.to(device), yb.to(device)
        logits  = model(xb)
        loss    = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        all_labels_list.extend(yb.cpu().numpy())

    train_acc = accuracy_score(all_labels_list, all_preds)
    avg_loss  = total_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['train_acc'].append(train_acc)

    # ── validate ──────────────────────────────────────────────────────────
    model.eval()
    val_preds, val_labels_list = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            val_preds.extend(torch.argmax(model(xb), 1).cpu().numpy())
            val_labels_list.extend(yb.numpy())

    val_acc = accuracy_score(val_labels_list, val_preds)
    val_f1  = f1_score(val_labels_list, val_preds, zero_division=0)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    if epoch in LOG_EPOCHS:
        print(f'[Epoch {epoch:02d}] loss={avg_loss:.4f} | '
              f'train_acc={train_acc*100:.2f}% | '
              f'val_acc={val_acc*100:.2f}% | '
              f'val_f1={val_f1:.4f} | '
              f'lr={scheduler.get_last_lr()[0]:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        if epoch in LOG_EPOCHS:
            print(f'  ✅ Best model saved (val_acc={val_acc*100:.2f}%)')

print(f'\n🏆  Best val acc: {best_val_acc*100:.2f}%')

Using device: cuda


Parameters: 1,176,177


Epoch 01:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 01:   2%|▏         | 1/54 [00:06<05:52,  6.65s/it]

Epoch 01:   4%|▎         | 2/54 [00:08<03:13,  3.73s/it]

...

Epoch 30:  94%|█████████▍| 51/54 [01:21<00:04,  1.60s/it]

Epoch 30:  96%|█████████▋| 52/54 [01:23<00:03,  1.64s/it]

Epoch 30:  98%|█████████▊| 53/54 [01:25<00:01,  1.63s/it]

Epoch 30: 100%|██████████| 54/54 [01:26<00:00,  1.57s/it]

[Epoch 30] loss=0.2340 | train_acc=90.42% | val_acc=86.30% | val_f1=0.7420 | lr=1.55e-09

🏆  Best val acc: 89.00%


## 6. Test Evaluation

In [8]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

test_preds, test_labels_list = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        test_preds.extend(torch.argmax(model(xb), 1).cpu().numpy())
        test_labels_list.extend(yb.numpy())

print('📊  Test Set Results')
print(f'   Accuracy  : {accuracy_score(test_labels_list, test_preds)*100:.2f}%')
print(f'   F1 Score  : {f1_score(test_labels_list, test_preds):.4f}')
print(f'   Precision : {precision_score(test_labels_list, test_preds):.4f}')
print(f'   Recall    : {recall_score(test_labels_list, test_preds):.4f}')
print()
print('Confusion Matrix:')
print(confusion_matrix(test_labels_list, test_preds))

📊  Test Set Results
   Accuracy  : 89.83%
   F1 Score  : 0.7754
   Precision : 0.6689
   Recall    : 0.9224

Confusion Matrix:
[[1662  200]
 [  34  404]]


## 7. Self-Supervised Contrastive Pre-Training

Implements the framework from the paper:
- **Gaussian Noise** augmentation: A1(x) = x + N(α, β²)
- **Horizontal Flipping** augmentation: A2(x) = reversed(x)
- **NT-Xent Contrastive Loss** (SimCLR-style): L(i,j) = -log[exp(sim(ti,tj)/η) / Σ exp(sim(ti,tk)/η)]
- **Projection Head** MLP to map backbone features to contrastive embedding space
- After pre-training, the backbone is frozen and a linear classifier is fine-tuned


In [9]:
# ─────────────────────────────────────────────────────────────────────────
# 7.1  Data Augmentation — Gaussian Noise & Horizontal Flipping
# ─────────────────────────────────────────────────────────────────────────

def augment_gaussian(x: torch.Tensor, alpha: float = 0.0, beta: float = 0.1) -> torch.Tensor:
    """
    A1(x) = x + M(x),  where M(x) ~ Gaussian(alpha, beta^2)
    x : (B, C, T)
    """
    noise = torch.randn_like(x) * beta + alpha
    return x + noise


def augment_flip(x: torch.Tensor) -> torch.Tensor:
    """
    A2(x) = x reversed along the time axis.
    A2(x) = X{n - φ + 1}  (horizontal flipping)
    x : (B, C, T)
    """
    return torch.flip(x, dims=[-1])


# Quick sanity check
_xtest = torch.randn(4, 1, 177)
print("Gaussian aug shape :", augment_gaussian(_xtest).shape)   # (4, 1, 177)
print("Flip aug shape     :", augment_flip(_xtest).shape)        # (4, 1, 177)
print("Flip reversal check:", torch.allclose(
    augment_flip(_xtest)[0, 0, :5],
    _xtest[0, 0, -5:].flip(0)
))


Gaussian aug shape : torch.Size([4, 1, 177])


Flip aug shape     : torch.Size([4, 1, 177])
Flip reversal check: True


In [10]:
# ─────────────────────────────────────────────────────────────────────────
# 7.2  Backbone Encoder (EEGformer without the classifier head)
# ─────────────────────────────────────────────────────────────────────────

class EEGformerEncoder(nn.Module):
    """
    The EEGformer backbone used as the feature extractor during contrastive
    pre-training.  Returns a flat feature vector per sample.

    Shape trace (C=1, kernel_size=5, T=177):
        ODCM  : (C=1, D=120, S=165)
        RTM   : (S=165, C+1=2, D=120)
        STM   : (C+1=2, S+1=166, D=120)
        TTM   : (M+1=7, C+1=2, S+1=166)
        flatten: (7 * 2 * 166) = 2324-d vector
    """
    def __init__(
        self,
        input_channels  = 1,
        time_steps      = 177,
        kernel_size     = 5,
        num_blocks      = 3,
        num_heads_rtm   = 6,
        num_heads_stm   = 6,
        num_heads_ttm   = 6,
        num_submatrices = 6,
    ):
        super().__init__()
        ncf = 120
        C, D = input_channels, ncf
        S   = time_steps - 3 * (kernel_size - 1)   # 165

        self.odcm = ODCM(C, kernel_size)
        self.rtm  = RTM((C, D, S),     num_blocks, num_heads_rtm)
        self.stm  = STM((S, C+1, D),   num_blocks, num_heads_stm)
        self.ttm  = TTM((C+1, S+1, D), num_submatrices, num_blocks, num_heads_ttm)

        # Compute flat feature size from TTM output shape (M+1, C+1, S+1)
        self.feat_dim = (num_submatrices + 1) * (C + 1) * (S + 1)   # 7*2*166=2324

    def forward_single(self, xi):
        """ Forward one sample xi : (C, T) → flat vector """
        xi = self.odcm(xi)      # (C, D, S)
        xi = self.rtm(xi)       # (S, C+1, D)
        xi = self.stm(xi)       # (C+1, S+1, D)
        xi = self.ttm(xi)       # (M+1, C+1, S+1)
        return xi.reshape(-1)   # flat

    def forward(self, x):
        """ x : (B, C, T)  →  (B, feat_dim) """
        return torch.stack([self.forward_single(x[i]) for i in range(x.shape[0])])


# Test encoder shape
_enc = EEGformerEncoder()
_out = _enc(torch.randn(2, 1, 177))
print(f"Encoder output shape: {_out.shape}")   # (2, 2324)
print(f"Feature dim         : {_enc.feat_dim}")


Encoder output shape: torch.Size([2, 2324])
Feature dim         : 2324


In [11]:
# ─────────────────────────────────────────────────────────────────────────
# 7.3  Projection Head  (SimCLR-style MLP on top of the backbone)
# ─────────────────────────────────────────────────────────────────────────

class ProjectionHead(nn.Module):
    """
    Two-layer MLP that maps backbone features to the contrastive embedding
    space: feat_dim → hidden_dim → proj_dim (with BN + ReLU between layers).
    """
    def __init__(self, feat_dim: int, hidden_dim: int = 256, proj_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, proj_dim),
        )

    def forward(self, h):
        """ h : (B, feat_dim)  →  (B, proj_dim) """
        return self.net(h)


# ─────────────────────────────────────────────────────────────────────────
# 7.4  NT-Xent Contrastive Loss
#      L(i,j) = -log[ exp(s(ti,tj)/η) / Σ_{k≠i} exp(s(ti,tk)/η) ]
#      where s(a,b) = cosine_similarity(a, b) ∈ [-1, 1]
# ─────────────────────────────────────────────────────────────────────────

class NTXentLoss(nn.Module):
    """
    NT-Xent (Normalized Temperature-scaled Cross-Entropy) loss.

    For a batch of N samples, each augmented twice, we have 2N vectors.
    For each anchor i its positive is the partner augmentation j=i+N (or i-N).
    The loss is averaged over all 2N anchors.
    """
    def __init__(self, temperature: float = 0.5):
        super().__init__()
        self.eta = temperature

    def forward(self, z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
        """
        z1, z2 : (N, proj_dim) — embeddings of aug-1 and aug-2 views.
        Returns scalar loss.
        """
        N = z1.shape[0]

        # L2-normalise → s(a,b) = a^T b
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        # Concatenate: rows 0..N-1 = aug1, rows N..2N-1 = aug2
        z   = torch.cat([z1, z2], dim=0)          # (2N, proj_dim)
        sim = z @ z.T / self.eta                   # (2N, 2N)

        # Mask out self-similarity on the diagonal
        mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
        sim  = sim.masked_fill(mask, -1e9)

        # Positive indices: for row i < N the positive is i + N, and vice-versa
        labels = torch.cat([
            torch.arange(N, 2 * N),
            torch.arange(0, N)
        ]).to(z.device)                            # (2N,)

        loss = F.cross_entropy(sim, labels)
        return loss


# Quick loss sanity check
_z1 = torch.randn(8, 128)
_z2 = torch.randn(8, 128)
_loss_fn = NTXentLoss(temperature=0.5)
print(f"NT-Xent loss on random embeddings: {_loss_fn(_z1, _z2):.4f}  (expect ≈ log(15)≈2.7)")


NT-Xent loss on random embeddings: 2.6924  (expect ≈ log(15)≈2.7)


In [12]:
# ─────────────────────────────────────────────────────────────────────────
# 7.5  Self-Supervised Contrastive Pre-Training Loop
# ─────────────────────────────────────────────────────────────────────────

SSL_EPOCHS    = 30          # pre-training epochs
SSL_LR        = 3e-4
SSL_BATCH     = 64          # smaller batch — each sample generates 2 views
SSL_TEMP      = 0.5         # temperature η in NT-Xent
PROJ_DIM      = 128         # projection head output dim
GAUSS_BETA    = 0.1         # std-dev for Gaussian noise augmentation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Build encoder + projection head ──────────────────────────────────────
ssl_encoder = EEGformerEncoder(
    input_channels=1, time_steps=177, kernel_size=5,
    num_blocks=3, num_heads_rtm=6, num_heads_stm=6,
    num_heads_ttm=6, num_submatrices=6,
).to(device)

proj_head = ProjectionHead(
    feat_dim   = ssl_encoder.feat_dim,
    hidden_dim = 256,
    proj_dim   = PROJ_DIM,
).to(device)

ssl_params = list(ssl_encoder.parameters()) + list(proj_head.parameters())
print(f'SSL trainable params: {sum(p.numel() for p in ssl_params if p.requires_grad):,}')

ssl_optimizer = torch.optim.AdamW(ssl_params, lr=SSL_LR, weight_decay=1e-4)
ssl_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    ssl_optimizer, T_max=SSL_EPOCHS, eta_min=1e-6
)
criterion_ssl = NTXentLoss(temperature=SSL_TEMP)

# ── Use only unlabelled X for pre-training (all 11500 samples) ───────────
# In self-supervised learning we intentionally ignore labels.
ssl_dataset = TensorDataset(X)   # X : (11500, 1, 177)
ssl_loader  = DataLoader(ssl_dataset, batch_size=SSL_BATCH, shuffle=True, drop_last=True)

# ── Pre-training loop ─────────────────────────────────────────────────────
LOG_SSL = set(range(1, SSL_EPOCHS + 1, 5)) | {SSL_EPOCHS}
ssl_history = []

print('\n🔁  Starting self-supervised contrastive pre-training ...')
for epoch in range(1, SSL_EPOCHS + 1):
    ssl_encoder.train()
    proj_head.train()

    epoch_loss = 0.0
    for (xb,) in tqdm(ssl_loader, desc=f'SSL Epoch {epoch:02d}', leave=False):
        xb = xb.to(device)              # (B, 1, 177)

        # ── Generate two augmented views ──────────────────────────────────
        xb_a1 = augment_gaussian(xb, alpha=0.0, beta=GAUSS_BETA)  # Gaussian noise view
        xb_a2 = augment_flip(xb)                                    # Horizontal flip view

        # ── Forward through encoder + projection head ─────────────────────
        h1 = ssl_encoder(xb_a1)   # (B, feat_dim)
        h2 = ssl_encoder(xb_a2)

        z1 = proj_head(h1)         # (B, proj_dim)
        z2 = proj_head(h2)

        # ── NT-Xent loss ──────────────────────────────────────────────────
        loss = criterion_ssl(z1, z2)

        ssl_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ssl_params, max_norm=1.0)
        ssl_optimizer.step()

        epoch_loss += loss.item()

    ssl_scheduler.step()
    avg = epoch_loss / len(ssl_loader)
    ssl_history.append(avg)

    if epoch in LOG_SSL:
        print(f'  [SSL Epoch {epoch:02d}]  contrastive loss = {avg:.4f}  '
              f'lr = {ssl_scheduler.get_last_lr()[0]:.2e}')

torch.save({
    'encoder': ssl_encoder.state_dict(),
    'proj_head': proj_head.state_dict(),
}, 'ssl_pretrained.pth')
print('\n✅  Pre-training complete. Checkpoint saved to ssl_pretrained.pth')


Device: cuda


SSL trainable params: 1,804,410

🔁  Starting self-supervised contrastive pre-training ...


SSL Epoch 01:   0%|          | 0/179 [00:00<?, ?it/s]

SSL Epoch 01:   1%|          | 1/179 [00:01<05:12,  1.75s/it]

SSL Epoch 01:   1%|          | 2/179 [00:03<04:42,  1.59s/it]

...

SSL Epoch 28:  44%|████▎     | 78/179 [01:50<02:27,  1.46s/it]

SSL Epoch 28:  44%|████▍     | 79/179 [01:52<02:24,  1.44s/it]

SSL Epoch 28:  45%|████▍     | 80/179 [01:53<02:23,  1.45s/it]

SSL Epoch 28:  45%|████▌     | 81/179 [01:55<02:21,  1.44s/it]

SSL Epoch 28:  46%|████▌     | 82/179 [01:56<02:19,  1.44s/it]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 7.6  Plot SSL Pre-training Loss Curve
# ─────────────────────────────────────────────────────────────────────────

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(ssl_history) + 1), ssl_history, marker='o', markersize=3,
         color='steelblue', linewidth=1.5, label='NT-Xent contrastive loss')
plt.xlabel('SSL Pre-training Epoch')
plt.ylabel('Loss')
plt.title('Self-Supervised Contrastive Pre-Training Loss')
plt.legend()
plt.tight_layout()
plt.savefig('ssl_pretrain_loss.png', dpi=150)
plt.show()
print('Loss curve saved.')


## 8. Linear Evaluation (Fine-tuning on Downstream Classification)

After pre-training, the backbone encoder weights are **frozen**.  
A lightweight linear classifier is placed on top of the frozen representations  
and trained on the labelled train split — following the standard linear evaluation protocol.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 8.1  Linear Classifier on Frozen SSL Encoder
# ─────────────────────────────────────────────────────────────────────────

class LinearClassifier(nn.Module):
    """Single linear layer on top of frozen encoder features."""
    def __init__(self, feat_dim: int, num_classes: int = 2):
        super().__init__()
        self.fc = nn.Linear(feat_dim, num_classes)

    def forward(self, h):
        return self.fc(h)


# Load pre-trained encoder
ckpt = torch.load('ssl_pretrained.pth', map_location=device)
ssl_encoder.load_state_dict(ckpt['encoder'])

# Freeze encoder
for p in ssl_encoder.parameters():
    p.requires_grad_(False)
ssl_encoder.eval()

linear_clf = LinearClassifier(ssl_encoder.feat_dim, num_classes=2).to(device)
print(f'Linear classifier params: {sum(p.numel() for p in linear_clf.parameters()):,}')

# ── Pre-compute frozen features for all splits ────────────────────────────
def extract_features(loader, encoder, device):
    """Extract frozen encoder features for a DataLoader."""
    feats, labels = [], []
    with torch.no_grad():
        for xb, yb in tqdm(loader, desc='Extracting features', leave=False):
            feats.append(encoder(xb.to(device)).cpu())
            labels.append(yb)
    return torch.cat(feats), torch.cat(labels)

print('\nExtracting features from frozen encoder ...')
train_feats, train_lbls = extract_features(train_loader, ssl_encoder, device)
val_feats,   val_lbls   = extract_features(val_loader,   ssl_encoder, device)
test_feats,  test_lbls  = extract_features(test_loader,  ssl_encoder, device)

print(f'  Train features: {train_feats.shape}')
print(f'  Val   features: {val_feats.shape}')
print(f'  Test  features: {test_feats.shape}')

# ── DataLoaders for feature tensors ───────────────────────────────────────
FINETUNE_BATCH = 256
train_feat_loader = DataLoader(TensorDataset(train_feats, train_lbls),
                               batch_size=FINETUNE_BATCH, shuffle=True)
val_feat_loader   = DataLoader(TensorDataset(val_feats,   val_lbls),
                               batch_size=FINETUNE_BATCH)
test_feat_loader  = DataLoader(TensorDataset(test_feats,  test_lbls),
                               batch_size=FINETUNE_BATCH)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 8.2  Fine-tune Linear Classifier
# ─────────────────────────────────────────────────────────────────────────

FINETUNE_EPOCHS = 20
FINETUNE_LR     = 1e-3

# Class-weighted loss (reuse class weights from supervised run)
ft_criterion = nn.CrossEntropyLoss(weight=class_wts)
ft_optimizer  = torch.optim.AdamW(linear_clf.parameters(), lr=FINETUNE_LR, weight_decay=1e-4)
ft_scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
    ft_optimizer, T_max=FINETUNE_EPOCHS, eta_min=1e-6
)

best_val_acc_ssl    = 0.0
best_clf_path       = 'best_ssl_linear_clf.pth'
ssl_finetune_history = {'train_acc': [], 'val_acc': [], 'val_f1': []}
LOG_FT = set(range(1, FINETUNE_EPOCHS + 1, 5)) | {FINETUNE_EPOCHS}

print('\n🔁  Fine-tuning linear classifier on frozen SSL features ...')
for epoch in range(1, FINETUNE_EPOCHS + 1):
    linear_clf.train()
    all_preds, all_lbls_list = [], []

    for hb, yb in train_feat_loader:
        hb, yb = hb.to(device), yb.to(device)
        logits = linear_clf(hb)
        loss   = ft_criterion(logits, yb)
        ft_optimizer.zero_grad()
        loss.backward()
        ft_optimizer.step()
        all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        all_lbls_list.extend(yb.cpu().numpy())

    ft_scheduler.step()
    tr_acc = accuracy_score(all_lbls_list, all_preds)
    ssl_finetune_history['train_acc'].append(tr_acc)

    # Validate
    linear_clf.eval()
    vp, vl = [], []
    with torch.no_grad():
        for hb, yb in val_feat_loader:
            vp.extend(torch.argmax(linear_clf(hb.to(device)), 1).cpu().numpy())
            vl.extend(yb.numpy())
    v_acc = accuracy_score(vl, vp)
    v_f1  = f1_score(vl, vp, zero_division=0)
    ssl_finetune_history['val_acc'].append(v_acc)
    ssl_finetune_history['val_f1'].append(v_f1)

    if epoch in LOG_FT:
        print(f'  [FT Epoch {epoch:02d}]  train_acc={tr_acc*100:.2f}%  '
              f'val_acc={v_acc*100:.2f}%  val_f1={v_f1:.4f}  '
              f'lr={ft_scheduler.get_last_lr()[0]:.2e}')

    if v_acc > best_val_acc_ssl:
        best_val_acc_ssl = v_acc
        torch.save(linear_clf.state_dict(), best_clf_path)

print(f'\n🏆  Best SSL val acc: {best_val_acc_ssl*100:.2f}%')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 8.3  Test Evaluation — SSL Model
# ─────────────────────────────────────────────────────────────────────────

linear_clf.load_state_dict(torch.load(best_clf_path, map_location=device))
linear_clf.eval()

tp, tl = [], []
with torch.no_grad():
    for hb, yb in test_feat_loader:
        tp.extend(torch.argmax(linear_clf(hb.to(device)), 1).cpu().numpy())
        tl.extend(yb.numpy())

ssl_acc  = accuracy_score(tl, tp)
ssl_f1   = f1_score(tl, tp, zero_division=0)
ssl_prec = precision_score(tl, tp, zero_division=0)
ssl_rec  = recall_score(tl, tp, zero_division=0)

print('📊  SSL Linear Evaluation — Test Set Results')
print(f'   Accuracy  : {ssl_acc*100:.2f}%')
print(f'   F1 Score  : {ssl_f1:.4f}')
print(f'   Precision : {ssl_prec:.4f}')
print(f'   Recall    : {ssl_rec:.4f}')
print()
print('Confusion Matrix:')
print(confusion_matrix(tl, tp))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 8.4  Comparison: Supervised vs SSL (Linear Evaluation)
# ─────────────────────────────────────────────────────────────────────────

# Retrieve supervised test results (run Cell 13 values or re-evaluate)
try:
    sup_acc  = accuracy_score(test_labels_list, test_preds)
    sup_f1   = f1_score(test_labels_list, test_preds, zero_division=0)
    sup_prec = precision_score(test_labels_list, test_preds, zero_division=0)
    sup_rec  = recall_score(test_labels_list, test_preds, zero_division=0)
except NameError:
    sup_acc = sup_f1 = sup_prec = sup_rec = float('nan')
    print('⚠  Run supervised training cells first to see comparison.')

metrics = ['Accuracy', 'F1 Score', 'Precision', 'Recall']
sup_vals = [sup_acc,  sup_f1,  sup_prec,  sup_rec]
ssl_vals = [ssl_acc,  ssl_f1,  ssl_prec,  ssl_rec]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width/2, sup_vals, width, label='Supervised (EEGformer)', color='steelblue')
b2 = ax.bar(x + width/2, ssl_vals, width, label='SSL + Linear Eval',      color='coral')

ax.set_ylabel('Score')
ax.set_title('Supervised vs Self-Supervised Contrastive Learning')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.legend()
ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=8)
ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=8)
plt.tight_layout()
plt.savefig('ssl_vs_supervised.png', dpi=150)
plt.show()

print('\nSummary table:')
print(f'  {"Metric":<12} {"Supervised":>12} {"SSL (Linear)":>14}')
print('  ' + '-'*40)
for m, sv, slv in zip(metrics, sup_vals, ssl_vals):
    print(f'  {m:<12} {sv:>12.4f} {slv:>14.4f}')
